# Chronos-2 trading features with native REINFORCE

This notebook shows the two Chronos workflows supported by `crosslearn`:

1. online extraction with `ChronosExtractor`
2. offline precomputation with `prepare_offline_dataframe`

`gym-anytrading` stays outside the package extras on purpose, so install it directly at the notebook level.


In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[chronos]" gym-anytrading pandas matplotlib      

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gym_anytrading.datasets import STOCKS_GOOGL
from gym_anytrading.envs import StocksEnv

from crosslearn import REINFORCE, make_vec_env
from crosslearn.extractors import ChronosExtractor, prepare_offline_dataframe

# Get gym-anytrading version from metadata
try:
    import importlib.metadata
    ga_version = importlib.metadata.version("gym-anytrading")
    print(f"gym-anytrading version: {ga_version}") #>> gym-anytrading version: 2.0.0
except importlib.metadata.PackageNotFoundError:
    print("gym-anytrading not found")

## Dataset and rolling-window setup

The raw environment emits rolling OHLCV windows. `ChronosExtractor` can embed those windows online, while `prepare_offline_dataframe(...)` can precompute the same representation into aligned dataframe columns ahead of training.


In [ ]:
LOOKBACK = 30
FEATURE_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
SELECTED_COLUMNS = ['Close', 'Volume']
FRAME_BOUND = (LOOKBACK, len(STOCKS_GOOGL))

display(STOCKS_GOOGL.head())
print('Frame bound:', FRAME_BOUND)
print('Feature columns:', FEATURE_COLUMNS)
print('Chronos-selected columns:', SELECTED_COLUMNS) 

In [ ]:
def online_process_data(env):
    start = env.frame_bound[0] - env.window_size
    end = env.frame_bound[1]
    prices = env.df.loc[:, 'Close'].to_numpy()[start:end]
    signal_features = env.df.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32)[start:end]
    return prices, signal_features


class OnlineStocksEnv(StocksEnv):
    _process_data = online_process_data


def make_online_env():
    return OnlineStocksEnv(
        df=STOCKS_GOOGL,
        window_size=LOOKBACK,
        frame_bound=FRAME_BOUND,
    )

sample_env = make_online_env()
print('Online observation space:', sample_env.observation_space)
sample_env.close()

## Online Chronos extraction

The environment still emits raw windows. `ChronosExtractor` performs the embedding step inside the policy so the agent can train directly on the encoded representation. This notebook keeps `n_envs=1` for clarity; on GPU, raise `n_envs` to build larger Chronos batches and use `use_async_env=True` only when environment stepping becomes the bottleneck.
            

In [ ]:
def render_single_episode(env, agent):
    obs, _ = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Info: {info}")
    env.unwrapped.render_all()
    plt.show()
    env.close()

In [ ]:
vec_env = make_vec_env(make_online_env, n_envs=1)

online_agent = REINFORCE(
    vec_env,
    n_steps=512,
    features_extractor_class=ChronosExtractor,
    features_extractor_kwargs={
        'feature_names': FEATURE_COLUMNS,
        'selected_columns': SELECTED_COLUMNS,
    },
    seed=42,
    verbose=2,
)
online_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_online_env(), online_agent)

## Offline Chronos embeddings

For offline mode, `prepare_offline_dataframe(...)` embeds the rolling windows once, appends `chronos_*` columns, and returns the trimmed aligned dataframe. Use `offline_df.filter(like="chronos_")` when wiring a second environment. Set `progress_bar=True` to monitor the offline embedding pass.


In [ ]:
offline_df = prepare_offline_dataframe(
    STOCKS_GOOGL,
    lookback=LOOKBACK,
    frame_bound=FRAME_BOUND,
    feature_columns=FEATURE_COLUMNS,
    selected_columns=SELECTED_COLUMNS,
    progress_bar=True,
)
display(offline_df.head())
display(offline_df.filter(like="chronos_").head())
print("Offline dataframe shape:", offline_df.shape)
print("Offline embedding columns:", offline_df.filter(like="chronos_").columns[:5].tolist(), "...")

In [ ]:
class OfflineStocksEnv(StocksEnv):
    def __init__(self, prices, signal_features, **kwargs):
        self._prices = prices
        self._signal_features = signal_features.astype(np.float32)
        super().__init__(**kwargs)

    def _process_data(self):
        return self._prices, self._signal_features


def make_offline_env():
    return OfflineStocksEnv(
        prices=offline_df["Close"].to_numpy(dtype=np.float32),
        signal_features=offline_df.filter(like="chronos_").to_numpy(dtype=np.float32),
        df=offline_df,
        window_size=1,
        frame_bound=(1, len(offline_df)),
    )

vec_offline = make_vec_env(make_offline_env, n_envs=1)
offline_agent = REINFORCE(vec_offline, n_steps=512, seed=42, verbose=2)
offline_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_offline_env(), offline_agent)

## Evaluate the Offline-Trained Agent in the Online Chronos Environment.
The agent trained on offline embeddings can be evaluated in the online environment without any code changes. The two Chronos workflows are designed to be interchangeable at the environment level, so you can train with one and test with the other seamlessly.

This is specially useful for real-world applications where you might want to precompute embeddings for faster training iterations, but still want to deploy the agent in an environment that computes embeddings on the fly.

In [ ]:
from crosslearn import REINFORCE, make_vec_env
from crosslearn.extractors import ChronosExtractor

# Assume you already trained the offline agent (as shown in the notebook)
#    (pre-compute embeddings once with prepare_offline_dataframe -> OfflineStocksEnv -> REINFORCE)
# offline_agent = REINFORCE(vec_offline, n_steps=512, seed=42, verbose=2)
# offline_agent.learn(total_timesteps=10_000)   # or however many you used

# 1.1. Define the *online* environment (raw rolling OHLCV windows) and
# 1.2. Create a fresh REINFORCE instance configured for *online* Chronos
#    (no learning — we only need the policy for inference)
online_agent = REINFORCE(
    make_vec_env(make_online_env, n_envs=1),   # online environment with raw windows
    n_steps=512,                               # dummy value, not used for eval
    features_extractor_class=ChronosExtractor,
    features_extractor_kwargs={
        "feature_names": FEATURE_COLUMNS,
        "selected_columns": SELECTED_COLUMNS,
        # match whatever you used in prepare_offline_dataframe (pooling, etc.)
    },
    seed=42,
    verbose=2,
)

# 2. Transfer the learned weights (MLP + actor head)
#    ChronosExtractor has no learnable parameters that conflict with the offline case
online_agent.policy.load_state_dict(
    offline_agent.policy.state_dict(),
    strict=False
)

In [ ]:
# 3. Evaluate / render in the online Chronos environment
online_env = make_online_env()
render_single_episode(online_env, online_agent)